# 05 — GATU vs ViraLift: 5-record head-to-head

Five fully-annotated PRRSV records provide the **truth** (their own GenBank annotation).
Both tools receive the **same stripped, sequence-only FASTA** (`targets/*.fasta`) and the **same
reference** (`reference_PQ623173.gb`); neither sees the truth annotation. Both are scored against
truth with the identical shared harness (IoU >= 0.90 or +-6 bp).

- **Truth source:** `PRRS_100seq_anno.gb` (the 5 records' own annotation) — read only for scoring.
- **Tool input:** `targets/<acc>.fasta` (annotation stripped) — what GATU and ViraLift both lift.
- **Reference:** `reference_PQ623173.gb` (PQ623173.1, 9 genes).

Records: `AF184212.1, AF325691.1, AY032626.1, AY262352.1, AY366525.1` (8 truth genes each, incl ORF5).

### Run order
1. **GATU** (manual, one at a time): load reference `reference_PQ623173.gb` + each `targets/<acc>.fasta`,
   accept the transferred annotations, and **save as GenBank** to `gatu_output/<accession>.gb`.
2. **ViraLift + validation**: run this notebook (needs `tblastn` on PATH). It lifts the 5 target
   FASTAs with ViraLift, ingests any GATU outputs present, and scores both. Re-run after adding
   GATU files.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'app').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd, matplotlib.pyplot as plt
from Bio import SeqIO
from app.validation._shared.validation_utils import *
from app.src.lifting.tblastn_lifter import lift_all_tblastn
from app.src.alias.gene_alias import apply_alias_to_features

CASE = Path('outputs/gatu_5case')
PICK = ['AF184212.1','AF325691.1','AY032626.1','AY262352.1','AY366525.1']
REF  = CASE / 'reference_PQ623173.gb'
bundle = load_reference_bundle(REF)

# TRUTH: the 5 records' own annotation (read only for scoring)
truth_src = {r.id: r for r in load_genbank_records(DATA/'PRRS'/'PRRS_100seq_anno.gb') if r.id in PICK}
# TOOL INPUT: the same stripped FASTA GATU receives (sequence only, no annotation)
target_records = {p.stem: SeqIO.read(str(p),'fasta') for p in (CASE/'targets').glob('*.fasta')}

# sanity: stripped FASTA == the annotated record's sequence (same input, annotation removed)
for acc in PICK:
    assert str(target_records[acc].seq) == str(truth_src[acc].seq), f'seq mismatch {acc}'
print('reference:', bundle['record'].id, '| genes:', [f['name'] for f in bundle['features']])
print('targets (tool input):', sorted(target_records))
print('truth records         :', sorted(truth_src))


In [ ]:
# Truth per record (record's own annotation, alias-normalised)
truth_by_acc = {acc: parse_truth_features(truth_src[acc], bundle['alias_lookup'], bundle['feature_type'])[0]
                for acc in PICK}
for acc in PICK:
    print(acc, '->', [t['name'] for t in truth_by_acc[acc]])

def score_one(acc, preds):
    df = compare_predictions_to_truth(preds, truth_by_acc[acc])
    df.insert(0, 'accession', acc)
    return df


In [ ]:
# ---- ViraLift (tblastn lift onto the SAME stripped target FASTA) ----
viralift_rows = []
for acc in PICK:
    lifted = lift_all_tblastn(bundle['features'], bundle['record'], target_records[acc],
                              validate_codons=(bundle['feature_type']=='CDS'))
    preds = lifted_to_rows(acc, lifted, 'viralift')
    viralift_rows.append(score_one(acc, preds))
viralift_df = pd.concat(viralift_rows, ignore_index=True)
print('ViraLift coord_correct: %d / %d' % (viralift_df['coord_correct'].sum(), len(viralift_df)))


In [ ]:
# ---- GATU ingest (from outputs/gatu_5case/gatu_output/<acc>.gb) ----
def load_gatu_preds(acc):
    hits = []
    for ext in ('gb','gbk','genbank','gbf'):
        hits += list((CASE/'gatu_output').glob(f'{acc}*.{ext}'))
    if not hits:
        return None
    rec = load_single_genbank(hits[0])
    feats = parse_features_for_type(rec, 'CDS')
    if not feats:
        return []
    feats = apply_alias_to_features(feats, bundle['alias_lookup'])  # normalise names like truth
    return [{'record_id':acc,'method':'gatu','pred_name':f['name'],
             'pred_start':f['start'],'pred_end':f['end'],'strand':f['strand'],
             'has_start_codon':None,'has_stop_codon':None,'in_frame':None} for f in feats]

gatu_rows, missing = [], []
for acc in PICK:
    preds = load_gatu_preds(acc)
    if preds is None:
        missing.append(acc); continue
    gatu_rows.append(score_one(acc, preds))

if missing:
    print('GATU output NOT yet present for:', missing)
    print('-> run GATU on these, save to outputs/gatu_5case/gatu_output/<acc>.gb, then re-run.')
gatu_df = pd.concat(gatu_rows, ignore_index=True) if gatu_rows else pd.DataFrame()
if len(gatu_df):
    print('GATU coord_correct: %d / %d' % (gatu_df['coord_correct'].sum(), len(gatu_df)))


In [ ]:
# ---- Side-by-side per (accession, gene) ----
def slim(df, tool):
    if not len(df): return pd.DataFrame(columns=['accession','pred_name',f'{tool}_start',f'{tool}_end',f'{tool}_correct'])
    return (df[['accession','pred_name','pred_start','pred_end','coord_correct']]
            .rename(columns={'pred_start':f'{tool}_start','pred_end':f'{tool}_end','coord_correct':f'{tool}_correct'}))

truth_long = pd.DataFrame([{'accession':a,'pred_name':t['name'],'truth_start':t['start'],'truth_end':t['end']}
                           for a in PICK for t in truth_by_acc[a]])
side = truth_long.merge(slim(viralift_df,'viralift'), on=['accession','pred_name'], how='left')
side = side.merge(slim(gatu_df,'gatu'), on=['accession','pred_name'], how='left')
side = side.sort_values(['accession','pred_name']).reset_index(drop=True)
side.to_csv(CASE/'side_by_side.tsv', sep='\t', index=False)
side


In [ ]:
# ---- Summary ----
def rate(df):
    return (int(df['coord_correct'].sum()), len(df), round(100*df['coord_correct'].mean(),1)) if len(df) else (0,0,None)
rows = [('ViraLift',)+rate(viralift_df)]
if len(gatu_df): rows.append(('GATU',)+rate(gatu_df))
summary = pd.DataFrame(rows, columns=['tool','coord_correct','total','coord_pct'])
summary.to_csv(CASE/'summary_5case.tsv', sep='\t', index=False)
print(summary.to_string(index=False))

o_v = viralift_df[viralift_df['pred_name']=='ORF5']
line = f"ORF5 -- ViraLift: {int(o_v['coord_correct'].sum())}/{len(o_v)}"
if len(gatu_df):
    o_g = gatu_df[gatu_df['pred_name']=='ORF5']
    line += f" | GATU: {int(o_g['coord_correct'].sum())}/{len(o_g)}"
print(line)


In [ ]:
# ---- Figure (drawn once GATU output is present) ----
if len(gatu_df):
    ax = summary.set_index('tool')['coord_pct'].plot(kind='bar', figsize=(5,4.2), color=['#2f7d4f','#b5651d'])
    ax.set_ylim(0,105); ax.set_ylabel('Coordinate correct (%)'); ax.set_xlabel('')
    ax.set_title('GATU vs ViraLift (5 PRRSV records, same truth)')
    for c in ax.containers: ax.bar_label(c, fmt='%.1f', padding=3)
    ax.figure.tight_layout(); ax.figure.savefig(CASE/'gatu_vs_viralift.png', dpi=180, bbox_inches='tight')
else:
    print('Add GATU outputs to draw the comparison figure.')
